## Multimodalni pristup

In [ ]:
import pandas as pd 
import joblib

df = pd.read_csv("../data/processed/final.csv")
model = joblib.load("../models/final_model.pkl")
threshold = joblib.load("../models/threshold.pkl")

In [2]:
df["target"] = df["final_result"].isin(["Fail", "Withdrawn"]).astype(int)
df["year"] = df["code_presentation"].astype(str).str[:4]

train_df = df[df["year"] == "2013"].copy()
test_df = df[df["year"] == "2014"].copy()

X_train = train_df.drop(columns=["final_result", "target", "id_student", "year"], errors="ignore")
y_train = train_df["target"]

X_test = test_df.drop(columns=["final_result", "target", "id_student", "year"], errors="ignore")
y_test = test_df["target"]

In [3]:
ml_risk = model.predict_proba(X_test)[:, 1]
ml_prediction = (ml_risk>= threshold).astype(int)
test_df["ml_risk_probability"] = ml_risk
test_df["ml_prediction"] = ml_prediction
test_df[["id_student", "code_module", "code_presentation", "final_result", "ml_risk_probability", "ml_prediction"]].head()

,id_student,code_module,code_presentation,final_result,ml_risk_probability,ml_prediction
358,6516,AAA,2014J,Pass,0.265252,0
359,24734,AAA,2014J,Pass,0.831003,1
360,26192,AAA,2014J,Distinction,0.050245,0
361,28061,AAA,2014J,Pass,0.182801,0
362,31600,AAA,2014J,Pass,0.327259,0


In [4]:
from transformers import pipeline

zero_shot_classifier = pipeline(
    "zero-shot-classification",
    model="joeddav/xlm-roberta-large-xnli"
)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
def napravi_tekstualni_opis(student):
    opis = f"""
    Student pohađa kurs {student['code_module']} u semestru {student['code_presentation']}.
    Broj prethodnih pokušaja: {student['num_of_prev_attempts']}.
    Broj kredita: {student['studied_credits']}.
    
    Prosečna TMA ocena studenta je {student["avg_score_TMA"]}.
    Student je predao {student["count_submitted_TMA"]} od ukupno {student["total_assessments_TMA"]} TMA zadataka.
    Stopa predaje TMA zadataka je {student["submission_rate_TMA"]}.
    Prosečan broj dana pre roka za TMA zadatke je {student["avg_days_before_deadline_TMA"]}.
    
    Prosečna CMA ocena studenta je {student["avg_score_CMA"]}.
    Student je predao {student["count_submitted_CMA"]} od ukupno {student["total_assessments_CMA"]} CMA zadataka.
    Stopa predaje CMA zadataka je {student["submission_rate_CMA"]}.
    Prosečan broj dana pre roka za CMA zadatke je {student["avg_days_before_deadline_CMA"]}.
    
    Ukupan broj klikova na VLE platformi je {student["total_clicks"]}.
    Poslednji dan aktivnosti studenta je {student["last_active_day"]}.
    Broj različitih aktivnosti na platformi je {student["unique_activities"]}.
    """

    return opis

In [6]:
student = test_df[test_df["ml_prediction"] == 1].iloc[0]
student_opis = napravi_tekstualni_opis(student)
print(student_opis)


    Student pohađa kurs AAA u semestru 2014J.
    Broj prethodnih pokušaja: 0.
    Broj kredita: 60.

    Prosečna TMA ocena studenta je 39.5.
    Student je predao 2.0 od ukupno 2.0 TMA zadataka.
    Stopa predaje TMA zadataka je 1.0.
    Prosečan broj dana pre roka za TMA zadatke je -7.0.

    Prosečna CMA ocena studenta je 0.0.
    Student je predao 0.0 od ukupno 0.0 CMA zadataka.
    Stopa predaje CMA zadataka je 0.0.
    Prosečan broj dana pre roka za CMA zadatke je 0.0.

    Ukupan broj klikova na VLE platformi je 189.0.
    Poslednji dan aktivnosti studenta je 86.0.
    Broj različitih aktivnosti na platformi je 6.0.
    


In [7]:
rizik = ["rizičan student", "uspešan student"]
hf_result = zero_shot_classifier( student_opis,candidate_labels=rizik)
hf_result

{'sequence': '\n    Student pohađa kurs AAA u semestru 2014J.\n    Broj prethodnih pokušaja: 0.\n    Broj kredita: 60.\n\n    Prosečna TMA ocena studenta je 39.5.\n    Student je predao 2.0 od ukupno 2.0 TMA zadataka.\n    Stopa predaje TMA zadataka je 1.0.\n    Prosečan broj dana pre roka za TMA zadatke je -7.0.\n\n    Prosečna CMA ocena studenta je 0.0.\n    Student je predao 0.0 od ukupno 0.0 CMA zadataka.\n    Stopa predaje CMA zadataka je 0.0.\n    Prosečan broj dana pre roka za CMA zadatke je 0.0.\n\n    Ukupan broj klikova na VLE platformi je 189.0.\n    Poslednji dan aktivnosti studenta je 86.0.\n    Broj različitih aktivnosti na platformi je 6.0.\n    ',
 'labels': ['rizičan student', 'uspešan student'],
 'scores': [0.5098960399627686, 0.4901040196418762]}

In [8]:
hf_scores = dict(zip(hf_result["labels"], hf_result["scores"]))
hf_risk_score = hf_scores["rizičan student"]
print("ML:", round(student["ml_risk_probability"], 4))
print("HF:", round(hf_risk_score, 4))

ML: 0.831
HF: 0.5099


In [9]:
multimodal_score = 0.7 * student["ml_risk_probability"] + 0.3 * hf_risk_score
multimodal_prediction = 1 if multimodal_score >= threshold else 0
print("Multimodal skor:", round(multimodal_score, 4))
print("Multimodal predvidjanje:", multimodal_prediction)

Multimodal skor: 0.7347
Multimodal predvidjanje: 1


In [10]:
from tqdm import tqdm
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score

test_sample = test_df.head(100).copy()
y_test_sample = y_test.head(100)

multimodal_scores = []

for _, student in tqdm(test_sample.iterrows(), total=len(test_sample)):
    opis = napravi_tekstualni_opis(student)
    hf_res = zero_shot_classifier(opis, candidate_labels=["rizičan student", "uspešan student"])
    hf_scores = dict(zip(hf_res["labels"], hf_res["scores"]))
    hf_risk = hf_scores["rizičan student"]
    
    multimodal = 0.7 * student["ml_risk_probability"] + 0.3 * hf_risk
    multimodal_scores.append(multimodal)

multimodal_scores = np.array(multimodal_scores)
multimodal_predictions = (multimodal_scores >= threshold).astype(int)

print("\nMultimodal rezultati (100 studenata):")
print(f"Accuracy: {accuracy_score(y_test_sample, multimodal_predictions):.4f}")
print(f"Precision: {precision_score(y_test_sample, multimodal_predictions):.4f}")
print(f"Recall: {recall_score(y_test_sample, multimodal_predictions):.4f}")
print(f"F1: {f1_score(y_test_sample, multimodal_predictions):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test_sample, multimodal_scores):.4f}")

print("\nML model rezultati (100 studenata):")
print(f"Accuracy: {accuracy_score(y_test_sample, test_sample['ml_prediction']):.4f}")
print(f"Precision: {precision_score(y_test_sample, test_sample['ml_prediction']):.4f}")
print(f"Recall: {recall_score(y_test_sample, test_sample['ml_prediction']):.4f}")
print(f"F1: {f1_score(y_test_sample, test_sample['ml_prediction']):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test_sample, test_sample['ml_risk_probability']):.4f}")


100%|██████████| 100/100 [02:48<00:00,  1.68s/it]


Multimodal rezultati (100 studenata):
Accuracy: 0.8500
Precision: 0.6400
Recall: 0.7273
F1: 0.6809
ROC AUC: 0.8683

ML model rezultati (100 studenata):
Accuracy: 0.8300
Precision: 0.6087
Recall: 0.6364
F1: 0.6222
ROC AUC: 0.8642
